In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

TISSUE = "crc"
df = pd.read_parquet("data/harmony_results.parquet")
df = df[df.tissue == TISSUE].copy()
STRAT = ["coverage", "quantile", "random"]
COLS = {"coverage": "#27ae60", "quantile": "#8e44ad", "random": "#7f7f7f"}
KSX = sorted(df[df.strategy == "coverage"].K.unique())
print(TISSUE, df.shape, "| held:", df.held.nunique(),
      "| cores:", df.drop_duplicates(["held", "dem_src", "normal_src"]).shape[0])
print(df.strategy.value_counts().to_dict())

In [ ]:
base = df[df.strategy.isin(["raw", "harmony_noref"])].groupby("strategy").transfer_auc.mean()
fig, ax = plt.subplots(figsize=(8.5, 5.5))
for s in STRAT:
    d = df[df.strategy == s]
    m = d.groupby("K").transfer_auc.mean().reindex(KSX)
    if s == "random":
        se = d.groupby("K").transfer_auc.sem().reindex(KSX)
        ax.plot(KSX, m, "-o", color=COLS[s], label=s)
        ax.fill_between(KSX, m - 1.96 * se, m + 1.96 * se, color=COLS[s], alpha=0.2)
    else:
        ax.plot(KSX, m, "-o", color=COLS[s], label=s)
ax.axhline(base["raw"], ls=":", color="black", label=f"raw {base['raw']:.3f}")
ax.axhline(base["harmony_noref"], ls="-.", color="0.6", label=f"harmony no-ref {base['harmony_noref']:.3f}")
ax.set_xlabel("paired studies added"); ax.set_ylabel("transfer AUC (mean over all cores x held)")
ax.set_title(f"{TISSUE}: lever averaged over every core and held"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
kmax = df.K.max()
sm = (df[df.strategy.isin(["coverage", "quantile"]) & (df.K == kmax)]
      .groupby(["held", "dem_src", "normal_src", "confound_severity"])
      .transfer_auc.max().reset_index(name="smart"))
rw = df[df.strategy == "raw"].groupby(["held", "dem_src", "normal_src"]).transfer_auc.mean().reset_index(name="raw")
hn = df[df.strategy == "harmony_noref"].groupby(["held", "dem_src", "normal_src"]).transfer_auc.mean().reset_index(name="harm0")
g = sm.merge(rw, on=["held", "dem_src", "normal_src"]).merge(hn, on=["held", "dem_src", "normal_src"])
g["gain_vs_raw"] = g.smart - g.raw
g["gain_vs_harm0"] = g.smart - g.harm0
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, col, ttl in [(axes[0], "gain_vs_raw", "smart@maxK - raw"),
                     (axes[1], "gain_vs_harm0", "smart@maxK - harmony(no ref)")]:
    ax.scatter(g.confound_severity, g[col], c=g.confound_severity, cmap="viridis", alpha=0.8)
    ax.axhline(0, color="0.7")
    ax.set_xlabel("confound severity (core A<->B batch distance)"); ax.set_ylabel(col)
    ax.set_title(f"{ttl}   (r={g[['confound_severity', col]].corr().iloc[0, 1]:.2f})")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
order = ["raw", "harmony_noref", "coverage", "quantile", "random"]
cmap = {"raw": "black", "harmony_noref": "0.6", "coverage": "#27ae60", "quantile": "#8e44ad", "random": "#c0392b"}
for ax, (xcol, ycol, xl, yl) in zip(axes, [
        ("batch_eta2", "celltype_eta2", "batch_eta2 (lower=batch removed)", "celltype_eta2 (higher=bio kept)"),
        ("batch_mix", "celltype_purity", "batch_mix kNN (higher=mixed)", "celltype_purity kNN (higher=kept)")]):
    for s in order:
        d = df[df.strategy == s]
        ax.scatter(d[xcol], d[ycol], s=14, alpha=0.4, color=cmap[s], label=s)
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.legend(fontsize=8)
axes[0].set_title(f"{TISSUE}: batch axis vs bio axis (eta^2)")
axes[1].set_title(f"{TISSUE}: cell-level scIB")
plt.tight_layout(); plt.show()

In [ ]:
rows = []
for s in ["raw", "harmony_noref", "coverage", "quantile", "random"]:
    d = df[df.strategy == s] if s in ("raw", "harmony_noref") else df[(df.strategy == s) & (df.K == df.K.max())]
    rows.append(dict(strategy=s, transfer_auc=d.transfer_auc.mean(), batch_eta2=d.batch_eta2.mean(),
                     celltype_eta2=d.celltype_eta2.mean(), batch_mix=d.batch_mix.mean(),
                     celltype_purity=d.celltype_purity.mean()))
print(pd.DataFrame(rows).round(3).to_string(index=False))